In [ ]:
import pandas as pd

df_artists = pd.read_csv('df_artists.csv')
df_songs   = pd.read_csv('df_songs.csv')
df_albums  = pd.read_csv('df_albums.csv')

print(f"df_artists: {df_artists.shape}")
print(f"df_songs:   {df_songs.shape}")
print(f"df_albums:  {df_albums.shape}")


df_artists: (14224, 54)
df_songs:   (5099, 35)
df_albums:  (4569, 35)


In [ ]:
# Import external Spotify song datasets for merge

import pandas as pd

# Downloaded from https://www.kaggle.com/datasets/suparnabiswas/billboard-hot-1002000-2023-data-with-features
df_suparnabiswas_billboard_hot_100_w_features = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/billboard_24years_lyrics_spotify.csv'
)

# Downloaded from https://www.kaggle.com/datasets/danield2255/data-on-songs-from-billboard-19992019
df_danield2255_spotify_song_attributes_1999_2019 = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/BillboardFromLast20/songAttributes_1999-2019.csv'
)

# Downloaded from https://www.kaggle.com/datasets/julianoorlandi/spotify-top-songs-and-audio-features
df_julianoorlandi_spotify_features = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/spotify_top_songs_audio_features.csv'
)

# Downloaded from https://www.kaggle.com/datasets/ziriantahirli/top5000-albums-on-spotify-data-analysis
df_ziriantahirli_top5000albums = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/top5000/Top5000.csv'
)

print(df_ziriantahirli_top5000albums.shape)
print(df_ziriantahirli_top5000albums.columns.tolist())
print()
print(df_ziriantahirli_top5000albums.head(3).to_string())
print(df_julianoorlandi_spotify_features.shape)
print(df_julianoorlandi_spotify_features.columns.tolist())
print()
print(df_julianoorlandi_spotify_features.head(3).to_string())
print(df_suparnabiswas_billboard_hot_100_w_features.shape)
print(df_suparnabiswas_billboard_hot_100_w_features.columns.tolist())
print()
print(df_danield2255_spotify_song_attributes_1999_2019.shape)
print(df_danield2255_spotify_song_attributes_1999_2019.columns.tolist())


(3397, 26)
['ranking', 'song', 'band_singer', 'songurl', 'titletext', 'url', 'year', 'lyrics', 'uri', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'type', 'id', 'track_href', 'analysis_url', 'duration_ms', 'time_signature']

(154931, 18)
['Unnamed: 0', 'Acousticness', 'Album', 'Artist', 'Danceability', 'Duration', 'Energy', 'Explicit', 'Instrumentalness', 'Liveness', 'Loudness', 'Mode', 'Name', 'Popularity', 'Speechiness', 'Tempo', 'TimeSignature', 'Valence']


In [12]:
df_youtube_links = pd.read_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_youtube_links.csv')
df_youtube_links.head()


,musicbrainz_artist_id,musicbrainz_artist_mbid,artist_name,youtube_url,youtube_id,url_type
0,1820772,d993169f-2033-4810-bae8-564d4aab89cd,$NOT,https://www.youtube.com/channel/UCNUL2JzmNpQJV...,UCNUL2JzmNpQJV4avV9AdjeA,channel_id
1,32589,9c0d337f-934c-45e9-a24e-c985b8059816,Krishna Das,https://music.youtube.com/channel/UCBSyRJHXHze...,UCBSyRJHXHze90Pf9QQdx9kw,channel_id
2,263470,fbeaf468-bc8d-40eb-a530-4f8982b2e1fe,The Audition,https://www.youtube.com/channel/UCFLt25X7v-DZE...,UCFLt25X7v-DZEiF9xXfVTrQ,channel_id
3,1259377,a4b4762e-2f1a-4bb8-8f2b-72388da8ab72,The Arcs,https://www.youtube.com/channel/UC2aPuO8f6fDj4...,UC2aPuO8f6fDj4BzHLe082zg,channel_id
4,211092,ca5220dc-96bf-4d58-aacb-5d1e0e6b0635,The Arbors,https://music.youtube.com/channel/UCta9tuyuXio...,UCta9tuyuXioO_TE7-6T9IuA,channel_id


In [15]:
# Create Wikipedia pageviews dataframe
# Data only available 2015-2024

import requests
import pandas as pd
import time
import re
import urllib.parse

# ── Load data ─────────────────────────────────────────────────
df_artists = pd.read_csv('df_artists.csv')

# MusicBrainz Wikipedia URLs (exact article titles)
df_wiki_mb = pd.read_csv('/tmp/artist_wikipedia_urls.csv')
df_wiki_mb['wiki_article'] = df_wiki_mb['wikipedia_url'].apply(
    lambda u: urllib.parse.unquote(u.split('/wiki/')[-1])
)
mb_lookup = dict(zip(df_wiki_mb['musicbrainz_artist_id'], df_wiki_mb['wiki_article']))

YEARS = list(range(2015, 2025))

# ── Helper functions ───────────────────────────────────────────
def get_annual_views(article, year):
    """Fetch total pageviews for a Wikipedia article in a given year."""
    article_enc = urllib.parse.quote(article, safe='')
    url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
           f'en.wikipedia/all-access/all-agents/{article_enc}/monthly/'
           f'{year}010100/{year}123100')
    r = requests.get(url, headers={'User-Agent': 'hitmaker-research/1.0'}, timeout=10)
    if r.status_code != 200:
        return None
    return sum(i['views'] for i in r.json().get('items', []))

def find_wikipedia_article(name):
    """Try to find a Wikipedia article for an artist by name search."""
    url = 'https://en.wikipedia.org/w/api.php'
    params = {
        'action': 'query', 'list': 'search',
        'srsearch': f'{name} musician', 'srlimit': 1, 'format': 'json'
    }
    r = requests.get(url, params=params,
                     headers={'User-Agent': 'hitmaker-research/1.0'}, timeout=10)
    if r.status_code != 200:
        return None
    results = r.json().get('query', {}).get('search', [])
    return results[0]['title'] if results else None

# ── Build lookup: name → (article, match_method) ──────────────
# Use MusicBrainz link where available, otherwise try name directly,
# then fall back to Wikipedia search
def resolve_article(row):
    mb_id = row['musicbrainz_artist_id']
    name  = row['name']

    # Method 1: MusicBrainz link
    if pd.notna(mb_id) and int(mb_id) in mb_lookup:
        return mb_lookup[int(mb_id)], 'musicbrainz_link'

    # Method 2: direct name (title-case)
    article_candidate = row['performer_pre_normalized'].replace(' ', '_')
    url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
           f'en.wikipedia/all-access/all-agents/'
           f'{urllib.parse.quote(article_candidate, safe="")}/monthly/20150101/20150131')
    r = requests.get(url, headers={'User-Agent': 'hitmaker-research/1.0'}, timeout=10)
    if r.status_code == 200:
        return article_candidate, 'name_direct'

    # Method 3: Wikipedia search API
    article = find_wikipedia_article(name)
    if article:
        return article, 'name_search'

    return None, 'not_found'

# ── Main batch loop ────────────────────────────────────────────
# Work on unique artists only
df_unique = df_artists.drop_duplicates(subset='name').copy()
print(f'Processing {len(df_unique)} unique artists...')

records = []
for i, row in df_unique.iterrows():
    if i % 50 == 0:
        print(f'  {i} / {len(df_unique)} ...')

    article, method = resolve_article(row)

    rec = {
        'name':               row['name'],
        'wiki_article':       article,
        'wiki_match_method':  method,
        'wiki_has_page':      article is not None,
    }

    if article:
        for yr in YEARS:
            views = get_annual_views(article, yr)
            rec[f'wiki_views_{yr}'] = views

        # Derived: views in first top-10 year
        top10_yr = row.get('first_year_top_10_songs')
        if pd.notna(top10_yr) and int(top10_yr) >= 2015:
            rec['wiki_views_first_top10_year'] = rec.get(f'wiki_views_{int(top10_yr)}')
        else:
            rec['wiki_views_first_top10_year'] = None

        # Derived: views in first charting year
        chart_yr = row.get('first_year_on_chart_songs')
        if pd.notna(chart_yr) and int(chart_yr) >= 2015:
            rec['wiki_views_first_charting_year'] = rec.get(f'wiki_views_{int(chart_yr)}')
        else:
            rec['wiki_views_first_charting_year'] = None
    else:
        for yr in YEARS:
            rec[f'wiki_views_{yr}'] = None
        rec['wiki_views_first_top10_year']    = None
        rec['wiki_views_first_charting_year'] = None

    records.append(rec)

# ── Save ───────────────────────────────────────────────────────
col_order = (['name', 'wiki_article', 'wiki_match_method'] +
             [f'wiki_views_{yr}' for yr in YEARS] +
             ['wiki_views_first_top10_year', 'wiki_views_first_charting_year',
              'wiki_has_page'])

df_wiki = pd.DataFrame(records)[col_order]
df_wiki.to_csv('df_wiki_pageviews.csv', index=False)

found = df_wiki['wiki_has_page'].sum()
print(f'\nDone. Found Wikipedia pages for {found}/{len(df_wiki)} artists ({found/len(df_wiki):.1%})')
print(df_wiki['wiki_match_method'].value_counts())


Processing 14224 unique artists...
  0 / 14224 ...
  50 / 14224 ...
  100 / 14224 ...
  150 / 14224 ...
  200 / 14224 ...
  250 / 14224 ...
  300 / 14224 ...
  350 / 14224 ...
  400 / 14224 ...
  450 / 14224 ...
  500 / 14224 ...
  550 / 14224 ...
  600 / 14224 ...
  650 / 14224 ...
  700 / 14224 ...
  750 / 14224 ...
  800 / 14224 ...
  850 / 14224 ...
  900 / 14224 ...
  950 / 14224 ...
  1000 / 14224 ...
  1050 / 14224 ...
  1100 / 14224 ...
  1150 / 14224 ...
  1200 / 14224 ...
  1250 / 14224 ...
  1300 / 14224 ...
  1350 / 14224 ...
  1400 / 14224 ...
  1450 / 14224 ...
  1500 / 14224 ...
  1550 / 14224 ...
  1600 / 14224 ...
  1650 / 14224 ...
  1700 / 14224 ...
  1750 / 14224 ...
  1800 / 14224 ...
  1850 / 14224 ...
  1900 / 14224 ...
  1950 / 14224 ...
  2000 / 14224 ...
  2050 / 14224 ...
  2100 / 14224 ...
  2150 / 14224 ...
  2200 / 14224 ...
  2250 / 14224 ...
  2300 / 14224 ...
  2350 / 14224 ...
  2400 / 14224 ...
  2450 / 14224 ...
  2500 / 14224 ...
  2550 / 14224 ...


ReadTimeout: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Read timed out. (read timeout=10)

In [16]:
# Resume Wikipedia pageviews collection from where the previous run stopped
# Saves existing records, then continues with unprocessed artists

# Save progress so far
if records:
    col_order = (['name', 'wiki_article', 'wiki_match_method'] +
                 [f'wiki_views_{yr}' for yr in YEARS] +
                 ['wiki_views_first_top10_year', 'wiki_views_first_charting_year',
                  'wiki_has_page'])
    df_wiki_partial = pd.DataFrame(records)[col_order]
    df_wiki_partial.to_csv('df_wiki_pageviews.csv', index=False)
    print(f"Saved {len(df_wiki_partial):,} records so far")

# Find which artists haven't been processed yet
already_done = {r['name'] for r in records}
df_remaining = df_unique[~df_unique['name'].isin(already_done)].copy()
print(f"Remaining: {len(df_remaining):,} artists")

# Resume
for i, (_, row) in enumerate(df_remaining.iterrows()):
    if i % 50 == 0:
        print(f'  {i} / {len(df_remaining)} ...')

    article, method = resolve_article(row)

    rec = {
        'name':              row['name'],
        'wiki_article':      article,
        'wiki_match_method': method,
        'wiki_has_page':     article is not None,
    }

    if article:
        for yr in YEARS:
            rec[f'wiki_views_{yr}'] = get_annual_views(article, yr)

        top10_yr = row.get('first_year_top_10_songs')
        if pd.notna(top10_yr) and int(top10_yr) >= 2015:
            rec['wiki_views_first_top10_year'] = rec.get(f'wiki_views_{int(top10_yr)}')
        else:
            rec['wiki_views_first_top10_year'] = None

        chart_yr = row.get('first_year_on_chart_songs')
        if pd.notna(chart_yr) and int(chart_yr) >= 2015:
            rec['wiki_views_first_charting_year'] = rec.get(f'wiki_views_{int(chart_yr)}')
        else:
            rec['wiki_views_first_charting_year'] = None
    else:
        for yr in YEARS:
            rec[f'wiki_views_{yr}'] = None
        rec['wiki_views_first_top10_year']    = None
        rec['wiki_views_first_charting_year'] = None

    records.append(rec)
    time.sleep(0.1)

# Save complete results
df_wiki = pd.DataFrame(records)[col_order]
df_wiki.to_csv('df_wiki_pageviews.csv', index=False)

found = df_wiki['wiki_has_page'].sum()
print(f'\nDone. Found Wikipedia pages for {found}/{len(df_wiki)} artists ({found/len(df_wiki):.1%})')
print(df_wiki['wiki_match_method'].value_counts())


Saved 8,527 records so far
Remaining: 5,697 artists
  0 / 5697 ...
  50 / 5697 ...
  100 / 5697 ...
  150 / 5697 ...
  200 / 5697 ...
  250 / 5697 ...
  300 / 5697 ...
  350 / 5697 ...
  400 / 5697 ...
  450 / 5697 ...
  500 / 5697 ...
  550 / 5697 ...
  600 / 5697 ...
  650 / 5697 ...
  700 / 5697 ...
  750 / 5697 ...
  800 / 5697 ...
  850 / 5697 ...
  900 / 5697 ...
  950 / 5697 ...
  1000 / 5697 ...
  1050 / 5697 ...
  1100 / 5697 ...
  1150 / 5697 ...
  1200 / 5697 ...
  1250 / 5697 ...
  1300 / 5697 ...
  1350 / 5697 ...
  1400 / 5697 ...
  1450 / 5697 ...
  1500 / 5697 ...
  1550 / 5697 ...
  1600 / 5697 ...
  1650 / 5697 ...
  1700 / 5697 ...
  1750 / 5697 ...
  1800 / 5697 ...
  1850 / 5697 ...
  1900 / 5697 ...
  1950 / 5697 ...
  2000 / 5697 ...
  2050 / 5697 ...
  2100 / 5697 ...
  2150 / 5697 ...
  2200 / 5697 ...
  2250 / 5697 ...
  2300 / 5697 ...
  2350 / 5697 ...
  2400 / 5697 ...
  2450 / 5697 ...
  2500 / 5697 ...
  2550 / 5697 ...
  2600 / 5697 ...
  2650 / 5697 ...

ReadTimeout: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Read timed out. (read timeout=10)

In [18]:
# Resume Wikipedia pageviews collection from where the previous run stopped
# Saves existing records, then continues with unprocessed artists

# Save progress so far
if records:
    col_order = (['name', 'wiki_article', 'wiki_match_method'] +
                 [f'wiki_views_{yr}' for yr in YEARS] +
                 ['wiki_views_first_top10_year', 'wiki_views_first_charting_year',
                  'wiki_has_page'])
    df_wiki_partial = pd.DataFrame(records)[col_order]
    df_wiki_partial.to_csv('df_wiki_pageviews.csv', index=False)
    print(f"Saved {len(df_wiki_partial):,} records so far")

# Find which artists haven't been processed yet
already_done = {r['name'] for r in records}
df_remaining = df_unique[~df_unique['name'].isin(already_done)].copy()
print(f"Remaining: {len(df_remaining):,} artists")

# Resume
for i, (_, row) in enumerate(df_remaining.iterrows()):
    if i % 50 == 0:
        print(f'  {i} / {len(df_remaining)} ...')

    try:
        article, method = resolve_article(row)
    except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
        print(f'    [timeout] {row["name"]!r} — skipping')
        article, method = None, 'timeout_error'

    rec = {
        'name':              row['name'],
        'wiki_article':      article,
        'wiki_match_method': method,
        'wiki_has_page':     article is not None,
    }

    if article:
        for yr in YEARS:
            try:
                rec[f'wiki_views_{yr}'] = get_annual_views(article, yr)
            except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError):
                rec[f'wiki_views_{yr}'] = None

        top10_yr = row.get('first_year_top_10_songs')
        if pd.notna(top10_yr) and int(top10_yr) >= 2015:
            rec['wiki_views_first_top10_year'] = rec.get(f'wiki_views_{int(top10_yr)}')
        else:
            rec['wiki_views_first_top10_year'] = None

        chart_yr = row.get('first_year_on_chart_songs')
        if pd.notna(chart_yr) and int(chart_yr) >= 2015:
            rec['wiki_views_first_charting_year'] = rec.get(f'wiki_views_{int(chart_yr)}')
        else:
            rec['wiki_views_first_charting_year'] = None
    else:
        for yr in YEARS:
            rec[f'wiki_views_{yr}'] = None
        rec['wiki_views_first_top10_year']    = None
        rec['wiki_views_first_charting_year'] = None

    records.append(rec)
    time.sleep(0.3)  # slightly longer to reduce timeout risk

# Save complete results
df_wiki = pd.DataFrame(records)[col_order]
df_wiki.to_csv('df_wiki_pageviews.csv', index=False)

found = df_wiki['wiki_has_page'].sum()
print(f'\nDone. Found Wikipedia pages for {found}/{len(df_wiki)} artists ({found/len(df_wiki):.1%})')
print(df_wiki['wiki_match_method'].value_counts())


Saved 13,053 records so far
Remaining: 1,171 artists
  0 / 1171 ...
  50 / 1171 ...
  100 / 1171 ...
  150 / 1171 ...
  200 / 1171 ...
  250 / 1171 ...
  300 / 1171 ...
  350 / 1171 ...
  400 / 1171 ...
  450 / 1171 ...
  500 / 1171 ...
  550 / 1171 ...
  600 / 1171 ...
  650 / 1171 ...
  700 / 1171 ...
  750 / 1171 ...
  800 / 1171 ...
  850 / 1171 ...
  900 / 1171 ...
  950 / 1171 ...
  1000 / 1171 ...
  1050 / 1171 ...
  1100 / 1171 ...
  1150 / 1171 ...

Done. Found Wikipedia pages for 3131/14224 artists (22.0%)
wiki_match_method
not_found           11093
name_search          2759
musicbrainz_link      372
Name: count, dtype: int64


In [20]:
df_wiki.head(20)

,name,wiki_article,wiki_match_method,wiki_views_2015,wiki_views_2016,wiki_views_2017,wiki_views_2018,wiki_views_2019,wiki_views_2020,wiki_views_2021,wiki_views_2022,wiki_views_2023,wiki_views_2024,wiki_views_first_top10_year,wiki_views_first_charting_year,wiki_has_page
0,!!! (chk chk chk),Chk Chk Boom,name_search,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52364.0,NaN,NaN,True
1,"""groove"" holmes",Richard Holmes (organist),name_search,5521.0,13328.0,11343.0,11360.0,10900.0,12930.0,12507.0,11918.0,12100.0,12206.0,NaN,NaN,True
2,"""little"" jimmy dickens",Little Jimmy Dickens,name_search,80716.0,85571.0,90653.0,76627.0,135798.0,118941.0,90642.0,77527.0,84670.0,66671.0,NaN,NaN,True
3,"""pookie"" hudson",List of Meerkat Manor meerkats,name_search,10340.0,22599.0,19816.0,19235.0,16577.0,24015.0,24111.0,16193.0,14253.0,13714.0,NaN,NaN,True
4,"""weird al"" yankovic",Bedrock Anthem,name_search,7585.0,16158.0,13703.0,13158.0,13546.0,14826.0,12884.0,16421.0,14423.0,13988.0,NaN,NaN,True
5,$not,Musician,name_search,92192.0,188167.0,205236.0,170312.0,180728.0,195954.0,209610.0,230040.0,196482.0,164158.0,NaN,209610.0,True
6,$uicideboy$,Suicideboys,name_search,NaN,958.0,1.0,202038.0,660137.0,603648.0,574872.0,654859.0,944224.0,767879.0,NaN,767879.0,True
7,$uicideboy$ x germ,Suicideboys,name_search,NaN,958.0,1.0,202038.0,660137.0,603648.0,574872.0,654859.0,944224.0,767879.0,NaN,NaN,True
8,& cardi b or nas,RuPaul's Drag Race season 18,name_search,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1242.0,NaN,NaN,True
9,& jack harlow,Jack Harlow,name_search,NaN,NaN,NaN,NaN,NaN,1317181.0,NaN,NaN,2380494.0,NaN,NaN,NaN,True


In [21]:
# Validate name_search matches by checking similarity between artist name and article title
# Nullifies matches where the article title is clearly unrelated to the artist name
import difflib, re

def normalize(s):
    """Lowercase, strip quotes/punctuation, collapse whitespace."""
    s = s.lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    return s.strip()

def name_match_score(artist_name, article_title):
    """Token-overlap ratio between normalized artist name and article title."""
    a_tokens = set(normalize(artist_name).split())
    b_tokens = set(normalize(article_title).split())
    # Remove very common filler words that don't help discriminate
    stopwords = {'the', 'a', 'an', 'of', 'and', 'in', 'on'}
    a_tokens -= stopwords
    b_tokens -= stopwords
    if not a_tokens:
        return 0.0
    overlap = a_tokens & b_tokens
    return len(overlap) / len(a_tokens)

# Compute scores for all name_search matches
mask_search = df_wiki['wiki_match_method'] == 'name_search'
df_wiki.loc[mask_search, '_match_score'] = df_wiki.loc[mask_search].apply(
    lambda r: name_match_score(r['name'], r['wiki_article']), axis=1
)

# Preview the distribution
print(df_wiki.loc[mask_search, '_match_score'].value_counts(bins=5, sort=False))
print()

# Show the worst matches for manual inspection
threshold = 0.3
suspect = df_wiki[mask_search & (df_wiki['_match_score'] < threshold)][
    ['name', 'wiki_article', '_match_score']
].sort_values('_match_score')
print(f"Suspect matches (score < {threshold}): {len(suspect)}")
print(suspect.to_string())


(-0.002, 0.2]     603
(0.2, 0.4]         98
(0.4, 0.6]        338
(0.6, 0.8]         76
(0.8, 1.0]       1644
Name: count, dtype: int64

Suspect matches (score < 0.3): 630
                                                                                               name                                                         wiki_article  _match_score
3                                                                                   "pookie" hudson                                       List of Meerkat Manor meerkats      0.000000
7186                                                                                  lindsey pavao                                   Black & White (Royal Tailor album)      0.000000
7188                                                              lindy conant & the circuit riders                                                       Gabriel Wilson      0.000000
7216                                                                                            

In [22]:
# Nullify all zero-similarity name_search matches, keep score > 0 for manual spot-check
view_cols = [f'wiki_views_{yr}' for yr in YEARS] + [
    'wiki_views_first_top10_year', 'wiki_views_first_charting_year'
]

bad = (df_wiki['wiki_match_method'] == 'name_search') & (df_wiki['_match_score'] == 0.0)
print(f"Nullifying {bad.sum()} score=0.0 matches")

df_wiki.loc[bad, 'wiki_article']      = None
df_wiki.loc[bad, 'wiki_match_method'] = 'rejected_low_similarity'
df_wiki.loc[bad, 'wiki_has_page']     = False
df_wiki.loc[bad, view_cols]           = None

# Review the borderline 0.0 < score < 0.3 cases separately
borderline = (df_wiki['_match_score'] > 0.0) & (df_wiki['_match_score'] < 0.3)
print(f"\nBorderline matches to review manually ({borderline.sum()}):")
print(df_wiki[borderline][['name', 'wiki_article', '_match_score']].to_string())


Nullifying 597 score=0.0 matches

Borderline matches to review manually (33):
                                                                                               name                  wiki_article  _match_score
59                                                                2cellos/london symphony orchestra                       2Cellos      0.250000
278                                                                          akon, lil wayne & niia                          Niia      0.250000
291                                                        al dimeola/john mclaughlin/paco de lucia                   Al Di Meola      0.142857
481                                                                       ana barbara/jennifer pena                 Jennifer Peña      0.250000
963    bas, jid, guapdad 4000, reese laflare, jace, mez, smokepurpp, buddy & ski mask the slump god                Buddy (rapper)      0.071429
2764                                                      

In [23]:
df_wiki.head(20)

,name,wiki_article,wiki_match_method,wiki_views_2015,wiki_views_2016,wiki_views_2017,wiki_views_2018,wiki_views_2019,wiki_views_2020,wiki_views_2021,wiki_views_2022,wiki_views_2023,wiki_views_2024,wiki_views_first_top10_year,wiki_views_first_charting_year,wiki_has_page,_match_score
0,!!! (chk chk chk),Chk Chk Boom,name_search,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52364.0,NaN,NaN,True,1.000000
1,"""groove"" holmes",Richard Holmes (organist),name_search,5521.0,13328.0,11343.0,11360.0,10900.0,12930.0,12507.0,11918.0,12100.0,12206.0,NaN,NaN,True,0.500000
2,"""little"" jimmy dickens",Little Jimmy Dickens,name_search,80716.0,85571.0,90653.0,76627.0,135798.0,118941.0,90642.0,77527.0,84670.0,66671.0,NaN,NaN,True,1.000000
3,"""pookie"" hudson",None,rejected_low_similarity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,0.000000
4,"""weird al"" yankovic",None,rejected_low_similarity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,0.000000
5,$not,None,rejected_low_similarity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,0.000000
6,$uicideboy$,None,rejected_low_similarity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,0.000000
7,$uicideboy$ x germ,None,rejected_low_similarity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,0.000000
8,& cardi b or nas,None,rejected_low_similarity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,0.000000
9,& jack harlow,Jack Harlow,name_search,NaN,NaN,NaN,NaN,NaN,1317181.0,NaN,NaN,2380494.0,NaN,NaN,NaN,True,1.000000


In [24]:
# Manually adjudicate the 33 borderline matches
keep_names = {
    'force m.d.\'s',
    'jr. walker & the all stars',
    '2cellos/london symphony orchestra',
    'jose luis rodriguez con los panchos',
    'mormon tabernacle choir/orchestra at temple square (wilberg)',
    'al dimeola/john mclaughlin/paco de lucia',
    'descemer bueno, zion & lennox or sean paul',
}

borderline_mask = (df_wiki['_match_score'] > 0.0) & (df_wiki['_match_score'] < 0.3)
nullify_borderline = borderline_mask & ~df_wiki['name'].isin(keep_names)
print(f"Nullifying {nullify_borderline.sum()} borderline matches, keeping {borderline_mask.sum() - nullify_borderline.sum()}")

df_wiki.loc[nullify_borderline, 'wiki_article']      = None
df_wiki.loc[nullify_borderline, 'wiki_match_method'] = 'rejected_low_similarity'
df_wiki.loc[nullify_borderline, 'wiki_has_page']     = False
df_wiki.loc[nullify_borderline, view_cols]           = None

df_wiki.drop(columns=['_match_score'], inplace=True)

found = df_wiki['wiki_has_page'].sum()
print(f"\nFinal: {found}/{len(df_wiki)} artists ({found/len(df_wiki):.1%})")
print(df_wiki['wiki_match_method'].value_counts())

df_wiki.to_csv('df_wiki_pageviews.csv', index=False)


Nullifying 26 borderline matches, keeping 7

Final: 2508/14224 artists (17.6%)
wiki_match_method
not_found                  11093
name_search                 2136
rejected_low_similarity      623
musicbrainz_link             372
Name: count, dtype: int64


In [27]:
df_wiki.sample(45)

,name,wiki_article,wiki_match_method,wiki_views_2015,wiki_views_2016,wiki_views_2017,wiki_views_2018,wiki_views_2019,wiki_views_2020,wiki_views_2021,wiki_views_2022,wiki_views_2023,wiki_views_2024,wiki_views_first_top10_year,wiki_views_first_charting_year,wiki_has_page
6912,lauren alaina,None,not_found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
6022,john roberts,John Roberts (musician),name_search,NaN,NaN,2521.0,NaN,NaN,NaN,NaN,3301.0,NaN,NaN,NaN,NaN,True
13816,we are harlot,None,not_found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
8965,patty smyth,None,not_found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1994,carla capuano,None,not_found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
13232,tokyo police club,None,not_found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
10211,samantha sang,None,not_found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
8972,paul baloche,None,not_found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
12681,the road,None,not_found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4004,famous dex,Famous Dex,name_search,NaN,NaN,1.0,NaN,NaN,192614.0,NaN,NaN,NaN,78948.0,NaN,1.0,True


In [28]:
# Check null counts per year to spot the pattern
has_page = df_wiki['wiki_has_page']
year_null_counts = {yr: df_wiki.loc[has_page, f'wiki_views_{yr}'].isna().sum() for yr in YEARS}
total = has_page.sum()
print(f"Null counts per year (among {total:,} artists with pages):")
for yr, n in year_null_counts.items():
    print(f"  {yr}: {n:,} nulls  ({n/total:.1%})")


Null counts per year (among 2,508 artists with pages):
  2015: 2,206 nulls  (88.0%)
  2016: 2,193 nulls  (87.4%)
  2017: 1,212 nulls  (48.3%)
  2018: 1,953 nulls  (77.9%)
  2019: 2,342 nulls  (93.4%)
  2020: 2,127 nulls  (84.8%)
  2021: 1,608 nulls  (64.1%)
  2022: 1,643 nulls  (65.5%)
  2023: 2,048 nulls  (81.7%)
  2024: 2,243 nulls  (89.4%)


In [30]:
# Re-fetch only null (artist, year) cells with retry and backoff
import time

def get_annual_views_robust(article, year, retries=3, backoff=5):
    article_enc = urllib.parse.quote(article, safe='')
    url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
           f'en.wikipedia/all-access/all-agents/{article_enc}/monthly/'
           f'{year}010100/{year}123100')
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, headers={'User-Agent': 'hitmaker-research/1.0'}, timeout=15)
            if r.status_code == 200:
                return sum(i['views'] for i in r.json().get('items', []))
            elif r.status_code == 429:
                wait = backoff * (2 ** attempt)
                print(f'    [429 rate limit] waiting {wait}s...')
                time.sleep(wait)
            else:
                return None  # 404 = article didn't exist that year
        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError):
            if attempt < retries:
                time.sleep(backoff)
            else:
                return None
    return None

has_page = df_wiki['wiki_has_page']
total_nulls = sum(df_wiki.loc[has_page, f'wiki_views_{yr}'].isna().sum() for yr in YEARS)
print(f'Refetching up to {total_nulls:,} null (artist, year) cells...')

filled = 0
for i, row in df_wiki[has_page].iterrows():
    null_years = [yr for yr in YEARS if pd.isna(row[f'wiki_views_{yr}'])]
    if not null_years:
        continue
    for yr in null_years:
        val = get_annual_views_robust(row['wiki_article'], yr)
        df_wiki.at[i, f'wiki_views_{yr}'] = val
        filled += 1
        time.sleep(0.2)  # change from 0.5

    if filled % 500 == 0:
        print(f'  {filled:,} / {total_nulls:,} filled...')
        df_wiki.to_csv('df_wiki_pageviews.csv', index=False)  # checkpoint

df_wiki.to_csv('df_wiki_pageviews.csv', index=False)
print(f'Done. Filled {filled:,} cells.')


Refetching up to 17,760 null (artist, year) cells...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
    [429 rate limit] waiting 5s...
  

KeyboardInterrupt: 